In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
ground_truth[10]

{'question': 'How do I join the office hours or live workshop if the Zoom link isn’t shared with students?',
 'document': '489dd1c9d9'}

In [3]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [4]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [5]:
q = ground_truth[10]
q

{'question': 'How do I join the office hours or live workshop if the Zoom link isn’t shared with students?',
 'document': '489dd1c9d9'}

In [6]:
doc_idx[q['document']]

{'id': '489dd1c9d9',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
 'answer': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'}

In [7]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [8]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
    course='llm-zoomcamp',
)

In [9]:
q['question']

'How do I join the office hours or live workshop if the Zoom link isn’t shared with students?'

In [10]:
answer = assistant.rag(q['question'])

In [11]:
assistant.total_cost()

0.0010170000000000001

In [12]:
print(answer)

The Zoom link is only shared with instructors/presenters/TAs, not students.

If you’re a student, you should:
- Watch the session live on the DataTalksClub YouTube Channel
- Check the announcements channel on Telegram or Slack for the live video URL before it starts
- Submit questions via Slido, not in the chat, since chat messages may be missed

If you can’t find the live link, follow the announcements in Telegram/Slack or watch on YouTube Live.


In [13]:
doc_id = q["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'

In [14]:
rag_result = {
    "question": q['question'],
    "answer_llm": answer,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'How do I join the office hours or live workshop if the Zoom link isn’t shared with students?',
 'answer_llm': 'The Zoom link is only shared with instructors/presenters/TAs, not students.\n\nIf you’re a student, you should:\n- Watch the session live on the DataTalksClub YouTube Channel\n- Check the announcements channel on Telegram or Slack for the live video URL before it starts\n- Submit questions via Slido, not in the chat, since chat messages may be missed\n\nIf you can’t find the live link, follow the announcements in Telegram/Slack or watch on YouTube Live.',
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalk

In [15]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [16]:
record = generate_rag_answer(q)
record

{'question': 'How do I join the office hours or live workshop if the Zoom link isn’t shared with students?',
 'answer_llm': 'Students don’t join the Zoom directly, since the Zoom link is only shared with instructors/presenters/TAs.\n\nInstead, you can:\n- Watch the session on the DataTalksClub YouTube channel\n- Find the live video URL in the announcements channel on Telegram or Slack before it starts\n- Submit questions via Slido, which is pinned in the chat during the live session\n\nDon’t rely on the Zoom chat for questions, since they may be missed.',
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post q

In [17]:
record = generate_rag_answer(q)
record

{'question': 'How do I join the office hours or live workshop if the Zoom link isn’t shared with students?',
 'answer_llm': 'The Zoom link is only shared with instructors, presenters, and TAs.\n\nAs a student, you should join the session via **YouTube Live**:\n- Check the **announcements channel on Telegram or Slack** for the video link before it starts.\n- You can also watch on the **DataTalksClub YouTube channel**.\n- If there’s a live Q&A, submit questions through **Slido** (the link is pinned in the chat when live).\n\nDon’t post questions in the chat, since they may be missed.',
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/Da

In [18]:
assistant.total_cost()

0.0030555

In [19]:
assistant.reset_usage()

In [20]:
assistant.total_cost()

0.0

In [21]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [22]:
from openai import RateLimitError
from tenacity import retry, wait_random_exponential, stop_after_attempt, retry_if_exception_type

@retry(
    retry=retry_if_exception_type(RateLimitError),
    wait=wait_random_exponential(min=1, max=60),
    stop=stop_after_attempt(6),
)
def _call_rag(question):
    return assistant.rag(question)


def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = _call_rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [23]:
with ThreadPoolExecutor(max_workers=2) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/590 [00:00<?, ?it/s]

In [24]:
results[:10]

[{'question': 'Can I still join the course if I found it late?',
  'answer_llm': 'Yes — you can still join the course if you found it late. If you want to receive a certificate, you’ll need to submit your project while submissions are still being accepted.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'Is it okay to start the course after it has already begun?',
  'answer_llm': 'Yes — you can start the course whenever you want, even after it has already begun. The videos and GitHub materials are available, and you can start learning and submitting homework while the submission form is open.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'If I join now, will I still be able to get a certificate?',
  'answer_llm': '

In [25]:
df_results = pd.DataFrame(results)

In [26]:
df_results.head()

,question,answer_llm,answer_orig,document
0,Can I still join the course if I found it late?,Yes — you can still join the course if you fou...,"Yes, but if you want to receive a certificate,...",74eb249bbf
1,Is it okay to start the course after it has al...,Yes — you can start the course whenever you wa...,"Yes, but if you want to receive a certificate,...",74eb249bbf
2,"If I join now, will I still be able to get a c...","Yes, but only if you finish the course with th...","Yes, but if you want to receive a certificate,...",74eb249bbf
3,Do I need to submit my project before submissi...,"Yes. To get the certificate, you need to submi...","Yes, but if you want to receive a certificate,...",74eb249bbf
4,What happens if I join the course after projec...,If the project submission form is no longer op...,"Yes, but if you want to receive a certificate,...",74eb249bbf


In [27]:
assistant.total_cost()

0.6516262500000002

In [28]:
df_results.to_csv("data/rag-answers-new.csv", index=False)

In [29]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [30]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [31]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [32]:
from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv()
openai_client = OpenAI()

In [33]:
import pandas as pd

df_answers = pd.read_csv("data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

In [34]:
rec = answers[0]
rec

{'question': 'Can I still join the course if I found it late?',
 'answer_llm': 'Yes — you can still join the course if you found it late. If you want to receive a certificate, you’ll need to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [35]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)
print(prompt)

Question:
Can I still join the course if I found it late?

Original Answer (ground truth):
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

AI Answer:
Yes — you can still join the course if you found it late. If you want to receive a certificate, you’ll need to submit your project while submissions are still being accepted.


In [36]:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning='The AI answer matches the ground truth: it says late joining is allowed and that certificate eligibility depends on submitting the project while submissions are still open. No key information is missing or altered.', score='good')

In [37]:
calc_price(usage)

{'input_cost': 0.00022425, 'output_cost': 0.0002385, 'total_cost': 0.00046275}

In [38]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [39]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the full meaning of the ground truth: late joining is allowed, and certificate eligibility depends on submitting the project before submissions close. It is semantically equivalent.', score='good')

In [40]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [41]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/590 [00:00<?, ?it/s]

In [42]:
results[10]

({'question': 'How do I join the office hours or live workshop if the Zoom link isn’t shared with students?',
  'document': '489dd1c9d9',
  'score': 'good',
  'reasoning': 'The AI answer matches the ground truth: it explains that the Zoom link is only for instructors/presenters/TAs, and students should watch via YouTube Live, use the announcements channel on Telegram/Slack to get the video URL, and submit questions through Slido rather than chat. No key information is missing or incorrect.'},
 ResponseUsage(input_tokens=455, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=81, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=536))

In [43]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [44]:
calc_total_price(usages)

0.4217205000000004

In [45]:
df_eval = pd.DataFrame(evaluations)

In [46]:
df_eval.head()

,question,document,score,reasoning
0,Can I still join the course if I found it late?,74eb249bbf,good,The AI answer matches the ground truth: it say...
1,Is it okay to start the course after it has al...,74eb249bbf,good,The AI answer preserves the core meaning: it s...
2,"If I join now, will I still be able to get a c...",74eb249bbf,good,The AI answer preserves the core condition tha...
3,Do I need to submit my project before submissi...,74eb249bbf,good,The AI answer preserves the key point that a p...
4,What happens if I join the course after projec...,74eb249bbf,good,The AI answer matches the core meaning of the ...


In [47]:
df_eval.score.value_counts()

score
good    565
bad      25
Name: count, dtype: int64

In [48]:
df_eval.score.value_counts(normalize=True)

score
good    0.957627
bad     0.042373
Name: proportion, dtype: float64

In [49]:
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
18,"Where do I find the lesson videos, code notebo...",04919992b3,bad,The AI answer gives the main repository and co...
32,Do I need to complete every homework assignmen...,9f689c185f,bad,The AI answer correctly states that homework i...
41,Do you know when the course is scheduled again?,bd31146b0e,bad,The ground truth gives a specific schedule tim...
43,Is there a planned upcoming session for this c...,bd31146b0e,bad,The ground truth states there is a planned upc...
46,Is there a main place I should start if I want...,31456f4b5f,bad,The ground truth says the main entry point sho...


In [50]:
df_eval.to_csv("data/rag-evaluations-new.csv", index=False)